## Week 2 Day 3

Now we get to more detail:

1. Different models

2. Structured Outputs

3. Guardrails

In [2]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, output_guardrail, GuardrailFunctionOutput
import os
from pydantic import BaseModel, Field
import asyncio
from agents import RunContextWrapper

In [3]:
load_dotenv(override=True)

True

In [4]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if  openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key not set (and this is optional)
OpenRouter API Key exists and begins sk-or-
Groq API Key exists and begins gsk_


In [5]:
instructions = """
You are a sales agent working for a retail company that sells women clothing. 
You write compelling sales emails that are likely to get a response. Do not make up any discount codes or promotions. Do not make up any product details. Only use the information provided in the prompt.
Do not make up any apointments or meetings. Write a fun, warm and sincere email. 
"""

### It's easy to use any models with OpenAI compatible endpoints in 3 steps:

STEP 1: Find the OpenAI compatible base URL (see Guide 9 in the guides folder)

In [6]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

STEP 2: Create a python client library instance (the async version)

In [7]:
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)
ollama_client = AsyncOpenAI(base_url="http://localhost:11434", api_key="anything")  # no API key needed for local LLaMA 2

STEP 3: Create a model object

In [8]:
gemini_model = OpenAIChatCompletionsModel(model="gemini-3.1-flash-lite", openai_client=gemini_client)
kimi_model = OpenAIChatCompletionsModel(model="moonshotai/kimi-k2.6", openai_client=openrouter_client)
oss_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)
ollama_model = OpenAIChatCompletionsModel(model="qwen2.5-coder:7b", openai_client=ollama_client)

In [132]:
sales_agent1 = Agent(name="Gemini Sales Agent", instructions=instructions, model=gemini_model)
sales_agent2 = Agent(name="Kimi Sales Agent", instructions=instructions, model=kimi_model)
sales_agent3 = Agent(name="GPT-OSS Sales Agent",instructions=instructions, model=oss_model)

In [133]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [134]:
result = await Runner.run(
    sales_agent3,
    input="Write a sales email for the best women winter socks in the world, with a variety of colors and very good warming factor to keep your feet warm in the coldest winter days. The socks are made of high-quality materials and are very comfortable to wear. The email should be fun, warm, and sincere, and should encourage the reader to buy the socks.",
)
print(result.final_output)

**Subject:** Keep Your Feet Cozy & Cheerful This Winter – Meet Our Best‑in‑Class Women’s Socks!  

Hi there,  

When the temperature drops and the wind whistles, there’s nothing quite like the comfort of a pair of socks that wrap your feet in pure warmth. We’re thrilled to introduce you to **the best women’s winter socks in the world**—a collection that’s as vibrant as it is cozy.

**Why you’ll love them**

- **A rainbow of colors** – From soft ivory and classic charcoal to playful berry and icy teal, there’s a shade to match every outfit and mood.  
- **Exceptional warming factor** – Engineered to keep your feet snug even on the coldest winter days, so you can step out with confidence.  
- **High‑quality materials** – Crafted from premium fibers that feel gentle on the skin while delivering long‑lasting durability.  
- **All‑day comfort** – A thoughtfully designed fit that stays in place without bunching, giving you that “just‑right” feel from sunrise to sunset.  

Picture yourself st

In [135]:
from messenger import send_email, push

USE_EMAIL = True

def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

In [136]:
send_message("Yet another test", "Hooray!", "<html><body><h1>Hooray!</h1></body></html>")

In [137]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

In [138]:
# tools = [tool1, tool2, tool3, send_email_tool]
tools = [tool3, send_email_tool]

In [139]:
# instructions = """
# You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
# """

# task = """
# Follow these steps:

# 1. Generate Drafts: Use each of the three sales_agent tools to generate different email drafts.
# Just instruct each to write a sales email; no further details are needed.
# Do not proceed until all three drafts are ready, one from each tool.
 
# 2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
# 3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
# 4. The number of available tools may change. Please make do with what is available. If only one tool is available, use it to generate the email and send it.
# """

# # sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="gpt-5.4-mini")
# sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=ollama_model)

In [140]:
instructions = """
You are a Sales Manager at retail company. Your goal is to find the single best cold sales email using the sales_agent tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use the sales_agent tools to generate email draft.
Just instruct them to write a sales email; no further details are needed.
Do not proceed until the draft is ready.
2. Evaluate: Review the draft.
3. Use your tool to send the the email to the user. Make sure that the email will be sent to the user. 
"""

# sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="gpt-5.4-mini")
sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=oss_model)


In [141]:
with trace("Sales Manager"):
    result = await Runner.run(sales_manager, task)
print(result.final_output)

The cold‑sales email draft has been generated and sent successfully. Let me know if you’d like any tweaks or a follow‑up message!


## Check out the trace

https://platform.openai.com/traces

## Part 2: Structured Outputs

An LLM produces text in natural language. But we can have it instead produce a "python object".

This is accomplished using the usual trickery: clever prompts & json!

1. We specify a Python object  
2. In the System prompt, the LLM is instructed to respond in JSON and follow a Schema which represents the Python object  
3. The LLM outputs JSON, and the framework populates a Python object based on it

When we specify the Python object, we create a subclass of BaseModel, which is part of the Pydantic framework.

Pydantic is a framework that easily allows defining a JSON schema and mapping between Python and json.

NOTES:

1. There is something about the way this is done that IS really clever - if you're interested, look up "constrained decoding".
2. Not all providers support Structured Outputs.


In [142]:
class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    number_of_sentences: int = Field(description="The number of sentences in the body of the email, not including the greeting and signature")
    contains_placeholders: bool = Field(description="Whether the email contains placeholders for personalization")

In [143]:
EmailReview.model_json_schema()

{'properties': {'is_professional': {'description': 'Whether the email is professional and appropriate',
   'title': 'Is Professional',
   'type': 'boolean'},
  'number_of_sentences': {'description': 'The number of sentences in the body of the email, not including the greeting and signature',
   'title': 'Number Of Sentences',
   'type': 'integer'},
  'contains_placeholders': {'description': 'Whether the email contains placeholders for personalization',
   'title': 'Contains Placeholders',
   'type': 'boolean'}},
 'required': ['is_professional',
  'number_of_sentences',
  'contains_placeholders'],
 'title': 'EmailReview',
 'type': 'object'}

In [144]:
email = """
Hi [Name],

I'm hitting you up to see if you'd like to buy our product. It's really great. You'll miss out if you don't buy it.


Laters.

Ed
"""

In [145]:
# checker = Agent(name="Checker", instructions="You review potential sales emails", model="gpt-5.4-mini", output_type=EmailReview)
checker_groq = Agent(name="Checker", instructions="You review potential sales emails", model=oss_model, output_type=EmailReview)
# result = await Runner.run(checker, email)
result = await Runner.run(checker_groq, email)

In [146]:
review = result.final_output
review

EmailReview(is_professional=False, number_of_sentences=3, contains_placeholders=True)

In [147]:
review.is_professional

False

## Part 3: Guardrails

Guardrails are extremely important in AgenticAI. Put simply, they are controls that you code either in logic or with another LLM call, to prevent undesirable behavior.

For me, the Guardrails impementation in OpenAI Agents SDK feels a bit like "framework voodoo". I suspect their motivation was to show framework-level controls to address this important topic.

But it's simple and clean to implement guardrails explicitly, as separate Runner.run() calls, or checks in your tool implementations.

Regardless - let's take a look at the framework tooling.

https://openai.github.io/openai-agents-python/guardrails/


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Watch out for a Gotcha</h2>
            <span style="color:#ff7800;">There are 3 types of Guardrails in OpenAI Agents SDK: input, output and tool. Input guardrails only run for the first input to the first Agent in Runner.run(). Output guardrails only run for the final output of the last agent. If you have guardrails on other agents, they will never be called.
            </span>
        </td>
    </tr>
</table>

In [148]:
@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(checker, message, context=ctx.context)
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional
    return GuardrailFunctionOutput(output_info={"review": review},tripwire_triggered=is_problem)

In [149]:
cowboy_instructions = instructions + "\nSpeak like a cowboy"

sales_agent_cowboy = Agent(name="Cowboy", instructions=cowboy_instructions, model=gemini_model, output_guardrails=[email_guardrail])

In [150]:
result = await Runner.run(sales_agent_cowboy_groq, "Write a cold sales email")
result.final_output

Error getting response: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}. (request_id: req_0943ce7ee31a4ddc8ec0a6876ef66e8b)


RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

Check out the trace:

https://platform.openai.com/traces

## On the other hand..

To state the obvious, this is simpler and will work in any framework

In [151]:
# simple_cowboy = Agent(name="Simple Cowboy", instructions=cowboy_instructions, model=gemini_model)
simple_cowboy_groq = Agent(name="Simple Cowboy Groq", instructions=cowboy_instructions, model=oss_model)
result = await Runner.run(simple_cowboy_groq, "Write a cold sales email")
email = result.final_output
print(email)


**Subject:** 🤠 Partner Up & Boost Your Store’s Bottom Line – No Bull, Just Results  

---

Howdy **[First Name]**,  

I’m **[Your Name]**, Sales Manager over at **[Your Retail Company]**. We’ve been riding the retail frontier for over a decade, helping storefronts like yours rope in more foot traffic, higher basket sizes, and repeat customers faster than a stallion on open range.

Here’s the low‑down on what we bring to the corral:

| What You’ll Get | How It Helps You |
|----------------|-----------------|
| **Curated, high‑margin product lines** – hand‑picked to match your customer profile | *More dollars per ticket* |
| **Turnkey merchandising kits** – signage, shelf‑talkers, and display guides | *Cut setup time in half* |
| **Data‑driven replenishment** – weekly sales intel so you never run out of the hot sellers | *Keep shelves stocked & sales soaring* |
| **Co‑branded marketing pushes** – email blasts, social shout‑outs, and in‑store events | *Drive traffic & brand buzz* |

All o

In [154]:
result = await Runner.run(checker_groq, email)
review = result.final_output
if not review.is_professional or review.contains_placeholders:
    print("The email is not professional or has placeholders and will not be sent")
else:
    print("Email is good")

BadRequestError: Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': ''}}

## Check out the trace:

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• Try different models<br/>• Add more input and output guardrails<br/>• Use structured outputs for the email generation
            </span>
        </td>
    </tr>
</table>

## OPTIONAL EXTRA: Sandbox Agents

This example will only work on Windows + WSL2, or Mac, or Linux

https://openai.github.io/openai-agents-python/sandbox_agents/

This is an execution harness - a runtime - "a persistent workspace where it can search large document sets, edit files, run commands, generate artifacts, and pick work back up from saved sandbox state."

You have to set up:
1. Manifest: the workspace
2. Capabilities: what it can do
3. SandboxRunConfig: where it runs

In [ ]:
from pathlib import Path
from agents.run import RunConfig
from agents.sandbox import Manifest, SandboxAgent, SandboxRunConfig, SandboxPathGrant
from agents.sandbox.capabilities import Capabilities
from agents.sandbox.entries import LocalDir
from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient

In [10]:
CODE_DIR = Path("code").resolve()
OUTPUT_DIR = Path("output").resolve()
if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir()

In [11]:
CODE_DIR

PosixPath('/home/tuna/Documents/agents/2_openai/code')

In [12]:
instructions = f"""
You are a software engineer that fixes bugs.
Review files in the sandbox code directory.

Write the fixed version of the file to this host output directory:
{OUTPUT_DIR}

Use full file paths when writing output.
Respond with a summary of what you did.
"""

In [14]:
manifest = Manifest(entries={"code": LocalDir(src=CODE_DIR)}, extra_path_grants=[SandboxPathGrant(path=str(OUTPUT_DIR))])
capabilities = Capabilities.default()
capabilities

[Filesystem(type='filesystem', session=None, run_as=None, configure_tools=None),
 Shell(type='shell', session=None, run_as=None, configure_tools=None),
 Compaction(type='compaction', session=None, run_as=None, policy=None)]

In [15]:
run_config = RunConfig(sandbox=SandboxRunConfig(client=UnixLocalSandboxClient()), workflow_name="Sandbox coding example")

In [16]:
# agent = SandboxAgent(name="Engineer", instructions=instructions, model="gpt-5.4-mini", default_manifest=manifest, capabilities=capabilities)
agent = SandboxAgent(name="Engineer", instructions=instructions, model=ollama_model, default_manifest=manifest, capabilities=capabilities)

In [17]:
result = await Runner.run(agent, "Fix the bug in the code", run_config=run_config)
print(result.final_output)

UserError: Hosted tools are not supported with the ChatCompletions API. Got tool type: <class 'agents.sandbox.capabilities.tools.apply_patch_tool.SandboxApplyPatchTool'>, tool: SandboxApplyPatchTool(name='apply_patch', description='Use the `apply_patch` tool to edit files. This is a FREEFORM tool, so do not wrap the patch in JSON.\nYour patch language is a stripped-down, file-oriented diff format designed to be easy to\nparse and safe to apply. You can think of it as a high-level envelope:\n\n*** Begin Patch\n[ one or more file sections ]\n*** End Patch\n\nWithin that envelope, you get a sequence of file operations.\nYou MUST include a header to specify the action you are taking.\nEach operation starts with one of three headers:\n\n*** Add File: <path> - create a new file. Every following line is a + line (the initial contents).\n*** Delete File: <path> - remove an existing file. Nothing follows.\n*** Update File: <path> - patch an existing file in place (optionally with a rename).\n\n

## OPTIONAL EXTRA: MCP Teaser!

In [170]:
from agents.mcp import MCPServerStreamableHttp


In [ ]:
task = """
In the new SandboxAgents feature in the OpenAI Agents SDK as of May 2026, what is the role of the Manifest object?
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
agent = Agent(name="Expert", instructions="Answer the question", model="gpt-4o-mini")
result = await Runner.run(agent, task)
print(result.final_output)

In [ ]:
params = {"url": "https://mcp.context7.com/mcp", "timeout": 60}


async with MCPServerStreamableHttp(name="Context7", params=params) as server:
    agent = Agent(name="Expert", instructions="Use Context7 to answer the question", mcp_servers=[server], model="gpt-4o-mini")
    result = await Runner.run(agent, task)

print(result.final_output)

And see the traces:

https://platform.openai.com/traces
